In [ ]:
"""Social-persona result figures (Chapter 4, Section: Social Personas).

Self-play, retry-3. Player 1 keeps the default behaviour; Player 2 is primed
with default / desperate / cunning. We report everything from P2's
(responder/buyer) perspective. Mirrors the cross-play notebook skin.
"""

In [ ]:
import sys, os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
def find_repo_root():
    for start in (Path(__file__).resolve().parent, Path.cwd().resolve()):
        for c in (start, *start.parents):
            if (c / ".logs").exists() and (c / "configs").exists() and (c / "_notebooks").exists():
                return c
    raise FileNotFoundError("repo root not found")

In [ ]:
ROOT_DIR = find_repo_root()
sys.path.insert(0, str(ROOT_DIR))
sys.path.insert(0, str(ROOT_DIR / "_notebooks" / "oss" / "style"))

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
import style
from style import wilson_ci, bootstrap_ci, errbars_from_ci
style.apply_thesis_style()

In [ ]:
# ---- Plotly-style thesis skin (copied from 1_cross_play_benchmark) ----------
matplotlib.rcParams["axes.prop_cycle"] = matplotlib.cycler(
    color=["#8C2D18", "#3D7A6A", "#B8A070", "#6B7E8C"])
matplotlib.rcParams["axes.edgecolor"] = "#8A8880"
matplotlib.rcParams["axes.labelcolor"] = "#3D3C38"
matplotlib.rcParams["xtick.color"] = "#3D3C38"
matplotlib.rcParams["ytick.color"] = "#3D3C38"
matplotlib.rcParams["grid.color"] = "#EAEDF0"
matplotlib.rcParams["grid.alpha"] = 0.9
matplotlib.rcParams["font.family"] = "sans-serif"
matplotlib.rcParams["font.sans-serif"] = ["Open Sans", "Noto Sans", "Liberation Sans", "DejaVu Sans"]
matplotlib.rcParams["mathtext.fontset"] = "dejavusans"
matplotlib.rcParams["axes.titlesize"] = 11
matplotlib.rcParams["axes.titleweight"] = "normal"
matplotlib.rcParams["axes.titlecolor"] = "#3D3C38"
matplotlib.rcParams["axes.titlepad"] = 10
matplotlib.rcParams["figure.titlesize"] = 11
matplotlib.rcParams["axes.linewidth"] = 0.8
matplotlib.rcParams["axes.grid.axis"] = "y"
matplotlib.rcParams["legend.frameon"] = False
matplotlib.rcParams["axes.unicode_minus"] = False   # render '-' with the sans glyph

In [ ]:
RED_LINE, RED_FILL = "#8C2D18", "#F5E8E4"
TEAL_LINE, TEAL_FILL = "#3D7A6A", "#E5F0ED"
SAND_LINE, SAND_FILL = "#B8A070", "#FAF5EE"
SLATE_LINE = "#6B7E8C"
ARROW_GRAY, PHASE_TEXT = "#8A8880", "#3D3C38"
FILL_OF = {RED_LINE: RED_FILL, TEAL_LINE: TEAL_FILL, SAND_LINE: SAND_FILL}
from matplotlib.colors import LinearSegmentedColormap
UTIL_CMAP = LinearSegmentedColormap.from_list("thesisSeq", [
    (0.00, "#F1E0D9"), (0.18, "#FBF6F3"), (0.42, "#E5F0ED"),
    (0.72, "#8FB8AC"), (1.00, "#3D7A6A")])

In [ ]:
FULL_WIDTH, HALF_WIDTH = style.FULL_WIDTH, style.HALF_WIDTH
LOGS_ROOT = str(ROOT_DIR / ".logs")
SIZES = ["very_small", "small", "medium"]
SIZE_LABEL = style.SIZE_LABEL                  # 4–9B / 12–14B / 24–27B
TIER_COLOR = dict(zip(SIZES, (RED_LINE, TEAL_LINE, SAND_LINE)))
PERSONAS = ["Default", "Desperate", "Cunning"]
GAMES = ["Trading", "Ultimatum", "BuySell"]

In [ ]:
FIG_DIR = ROOT_DIR / "_notebooks" / "oss" / "figures" / "2_social_personas"
FIG_DIR.mkdir(parents=True, exist_ok=True)
THESIS_DIR = ROOT_DIR / "context" / "MSc_Thesis" / "figures" / "results" / "personas"
THESIS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def save_plot(fig, slug):
    """Save without the descriptive whole-figure title (caption supplies it);
    keep panel/facet titles. Copy the PNG into the thesis tree as well."""
    hidden = []
    sup = getattr(fig, "_suptitle", None)
    if sup is not None and sup.get_text():
        sup.set_visible(False); hidden.append(sup)
    real = [a for a in fig.axes if a.get_label() != "<colorbar>"]
    if len(real) == 1 and real[0].get_title():
        real[0].title.set_visible(False); hidden.append(real[0].title)
    out = FIG_DIR / f"{slug}.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    fig.savefig(THESIS_DIR / f"{slug}.png", dpi=300, bbox_inches="tight")
    for a in hidden:
        a.set_visible(True)
    return out

In [ ]:
# --------------------------------------------------------------------------- data
BEH = {
    "Trading": {
        "": "Default",
        "\nYou are cunning and sly in your plan to get more resources. Humiliate your opponent as much as possible. Insult them.": "Cunning",
        "\nYou must fake being desperate. Supplicate and beg to get more resources.": "Desperate"},
    "Ultimatum": {
        "": "Default",
        "\nYou are cunning and sly in your plan to get more than your opponent. Humiliate your opponent as much as possible. Insult them.": "Cunning",
        "\nYou must fake being desperate. Supplicate and beg to get more dollars.": "Desperate"},
    "BuySell": {
        "": "Default",
        "\nYou are cunning and sly in your plan to spend less ZUP. Humiliate your opponent as much as possible. Insult them.": "Cunning",
        "\nYou must fake being desperate. Supplicate and beg to spend less ZUP.": "Desperate"},
}
GAME_DIR = {"Trading": "trading", "Ultimatum": "ultimatum", "BuySell": "buysell"}

In [ ]:
def _clean(m):
    return m.split("/")[-1].replace("-Instruct", "").replace("-instruct", "")

In [ ]:
def _rv(res):
    return sum(res["_value"].values())

In [ ]:
def load_game(game, size):
    """Return rows with P1/P2 payoff, win flags, and no-deal flag (completed games)."""
    d = os.path.join(LOGS_ROOT, f"section_two/{GAME_DIR[game]}_section_two_personas_retry3/{size}")
    rows = []
    for root, _, files in os.walk(d):
        if "game_state.json" not in files:
            continue
        try:
            data = json.load(open(os.path.join(root, "game_state.json")))
            last = data["game_state"][-1]
            if last.get("current_iteration") != "END":
                continue
            settings = data["game_state"][0]["settings"]
            soc = settings.get("player_social_behaviour", ["", ""])[1]
            persona = BEH[game].get(soc, soc)
            s = last["summary"]
            if game == "BuySell":
                o = s["player_outcome"]
                p1, p2 = float(o[0]), float(o[1])
                no_deal = (p1 == 0 and p2 == 0)
            else:
                p1 = _rv(s["final_resources"][0]) - _rv(s["initial_resources"][0])
                p2 = _rv(s["final_resources"][1]) - _rv(s["initial_resources"][1])
                if game == "Ultimatum":
                    p1 = p1 + 100
                    if p1 == 100:
                        p1 = 0
                no_deal = (p1 == 0 and p2 == 0)
            rows.append({"model": _clean(data["players"][0].get("model_id", data["players"][0].get("model"))),
                         "persona": persona, "p1": p1, "p2": p2,
                         "win2": p2 > p1, "win1": p1 > p2, "no_deal": no_deal})
        except Exception:
            pass
    df = pd.DataFrame(rows)
    df["size"] = size
    df["game"] = game
    return df

In [ ]:
ALL = pd.concat([load_game(g, s) for g in GAMES for s in SIZES], ignore_index=True)

In [ ]:
def cell(game, size, persona):
    return ALL[(ALL.game == game) & (ALL["size"] == size) & (ALL.persona == persona)]

In [ ]:
# --------------------------------------------------------------------------- bars
def grouped_bars(ax, value_fn, ci_fn=None, ylim=None, ylabel=""):
    """Persona on x (3 groups), one bar per tier; tinted fill + coloured border."""
    n_t = len(SIZES)
    width = 0.78 / n_t
    x = np.arange(len(PERSONAS))
    for ti, size in enumerate(SIZES):
        centers, los, his = [], [], []
        for persona in PERSONAS:
            v, ci = value_fn(game_ctx, size, persona)
            centers.append(v)
            if ci is not None:
                los.append(ci[0]); his.append(ci[1])
        pos = x + (ti - (n_t - 1) / 2) * width
        col = TIER_COLOR[size]
        ax.bar(pos, centers, width=width, color=col, edgecolor="white",
               linewidth=0.5, label=SIZE_LABEL[size], zorder=3)
        if ci_fn is not None and los:
            yerr = errbars_from_ci(centers, list(zip(los, his)))
            ax.errorbar(pos, centers, yerr=yerr, fmt="none", ecolor=ARROW_GRAY,
                        elinewidth=0.9, capsize=2.2, zorder=4)
    ax.set_xticks(x, PERSONAS)
    ax.tick_params(axis="x", labelsize=8)
    if ylim:
        ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel)

In [ ]:
# value functions ------------------------------------------------------------
def win_rate_val(game, size, persona):
    c = cell(game, size, persona)
    k = int(c.win2.sum()); n = int(c.win2.sum() + c.win1.sum())
    if n == 0:
        return np.nan, None
    return k / n, wilson_ci(k, n)

In [ ]:
def no_deal_val(game, size, persona):
    c = cell(game, size, persona)
    n = len(c)
    if n == 0:
        return np.nan, None
    k = int(c.no_deal.sum())
    return k / n, wilson_ci(k, n)

In [ ]:
def payoff_val(game, size, persona):
    c = cell(game, size, persona)
    if c.empty:
        return np.nan, None
    return c.p2.mean(), bootstrap_ci(c.p2.values)

In [ ]:
# ----------------------------------------------------------- Figure 1: win rate
def fig_winrate():
    fig, axs = plt.subplots(1, 3, figsize=(FULL_WIDTH, 2.5), constrained_layout=True)
    global game_ctx
    for ax, game in zip(axs, GAMES):
        game_ctx = game
        grouped_bars(ax, win_rate_val, ci_fn=True, ylim=(0, 1.05))
        ax.axhline(0.5, color=ARROW_GRAY, lw=0.7, ls=(0, (4, 3)), zorder=1)
        ax.set_title(game)
    axs[0].set_ylabel("P2 win rate (decisive)")
    h, l = axs[0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.12),
               title="Tier", title_fontsize=9, fontsize=8)
    fig.suptitle("Persona win rate by game and tier")
    save_plot(fig, "persona_winrate_by_game")
    plt.close(fig)

In [ ]:
# --------------------------------------------------------- Figure 2: no-deal rate
def fig_nodeal():
    fig, axs = plt.subplots(1, 3, figsize=(FULL_WIDTH, 2.5), constrained_layout=True)
    global game_ctx
    for ax, game in zip(axs, GAMES):
        game_ctx = game
        grouped_bars(ax, no_deal_val, ci_fn=True, ylim=(0, 1.05))
        ax.set_title(game)
    axs[0].set_ylabel("No-deal rate")
    h, l = axs[0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.12),
               title="Tier", title_fontsize=9, fontsize=8)
    fig.suptitle("No-deal rate by persona")
    save_plot(fig, "persona_no_deal_rate")
    plt.close(fig)

In [ ]:
# ----------------------------------------------------------- Figure 3: payoff
def fig_payoff():
    units = {"Trading": "P2 resource Δ", "Ultimatum": "P2 dollars", "BuySell": "buyer profit"}
    fig, axs = plt.subplots(1, 3, figsize=(FULL_WIDTH, 2.5), constrained_layout=True)
    global game_ctx
    for ax, game in zip(axs, GAMES):
        game_ctx = game
        grouped_bars(ax, payoff_val, ci_fn=True)
        ax.axhline(0, color=ARROW_GRAY, lw=0.8, zorder=2)
        ax.set_title(game)
        ax.set_ylabel(units[game])
    h, l = axs[0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.12),
               title="Tier", title_fontsize=9, fontsize=8)
    fig.suptitle("Mean P2 payoff by persona")
    save_plot(fig, "persona_payoff_by_game")
    plt.close(fig)

In [ ]:
# ------------------------------------------------- Figure 4: win-rate delta heatmap
def fig_delta_heatmap():
    rows, labels = [], []
    for game in GAMES:
        for size in SIZES:
            base, _ = win_rate_val(game, size, "Default")
            d, _ = win_rate_val(game, size, "Desperate")
            c, _ = win_rate_val(game, size, "Cunning")
            rows.append([(d - base) * 100, (c - base) * 100])
            labels.append(f"{game} · {SIZE_LABEL[size]}")
    mat = np.array(rows)
    fig, ax = plt.subplots(figsize=(HALF_WIDTH + 0.6, 4.0), constrained_layout=True)
    style.heatmap(ax, mat, labels, ["Desperate", "Cunning"],
                  fmt="{:+.0f}", cmap=UTIL_CMAP)
    ax.set_title("Δ P2 win rate vs default (pp)")
    save_plot(fig, "persona_winrate_delta_heatmap")
    plt.close(fig)

In [ ]:
fig_winrate()
fig_nodeal()
fig_payoff()
fig_delta_heatmap()

In [ ]:
# --------------------------------------------------------------------------- table
print("\n===== SUMMARY TABLE NUMBERS (P2/buyer, decisive win rate; mean payoff) =====")
for game in GAMES:
    print(f"\n{game}")
    for size in SIZES:
        parts = []
        for persona in PERSONAS:
            wr, _ = win_rate_val(game, size, persona)
            pf, _ = payoff_val(game, size, persona)
            nd, _ = no_deal_val(game, size, persona)
            parts.append(f"{persona[:4]} wr={wr*100:4.0f} pf={pf:5.1f} nd={nd*100:3.0f}")
        print(f"  {SIZE_LABEL[size]:7s} | " + " | ".join(parts))

In [ ]:
print("\nSaved figures to:", FIG_DIR)
print("Copied to:", THESIS_DIR)